# AI Agent × RustGIS：從自然語言到可驗證的地震 GIS
Python 3.12+；依序執行。預設 replay 模式不需金鑰，重播固定案例的規劃（不是即時 AI）。live 模式使用 OpenAI API，可能計費，金鑰放 Colab Secrets 的 OPENAI_API_KEY，不要寫進程式或 GitHub。模型只接收問題和欄位規格，不接收金鑰或完整資料。此教材為受限的規劃-執行 agent，不是原生 MCP。
來源：[RustGIS Python](https://rustgis.app/python/)、[MCP](https://github.com/opengeos/RustGIS/blob/main/docs/mcp.md)、[USGS FDSN](https://earthquake.usgs.gov/fdsnws/event/1/)。尚未實際執行。

In [ ]:
%pip install -q rustgis geopandas requests openai matplotlib

In [ ]:
# @title 自然語言問題
mode = "replay" # @param ["replay","live"]
question = "請分析2024年4月臺灣附近規模5以上、深度小於30公里的地震，建立50公里環域，並統計地震數量。" # @param {type:"string"}
model = "gpt-4.1-mini" # @param {type:"string"}
# replay 只支援上述固定問題；修改問題請切換 live。
DEFAULT_QUESTION = "請分析2024年4月臺灣附近規模5以上、深度小於30公里的地震，建立50公里環域，並統計地震數量。"
import json, math
from pathlib import Path
from datetime import date
import requests
import pandas as pd
import geopandas as gpd
import rustgis
from shapely.geometry import box
from IPython.display import display, Image
out = Path("agent_outputs")
out.mkdir(exist_ok=True)

## 1 規劃工具：模型只能提出固定欄位的 JSON，不得生成可執行 Python 或任意 SQL。日期以 UTC 表示，區域固定 119–123.5°E、21.5–26°N。

In [ ]:
schema = {"type":"object","properties":{
    "start":{"type":"string"},"end":{"type":"string"},
    "min_mag":{"type":"number"},"max_depth_km":{"type":"number"},
    "buffer_km":{"type":"number"}},
    "required":["start","end","min_mag","max_depth_km","buffer_km"],
    "additionalProperties":False}
if mode == "replay":
    if question != DEFAULT_QUESTION:
        raise ValueError("replay 僅支援固定案例；請使用 live 或恢復預設問題")
    plan = dict(start="2024-04-01",end="2024-05-01",min_mag=5,max_depth_km=30,buffer_km=50)
else:
    from google.colab import userdata
    from openai import OpenAI
    client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
    response = client.chat.completions.create(model=model,
        messages=[{"role":"system","content":"將問題轉成指定 JSON。只支援臺灣固定區域地震查詢、最低規模（>=）、最大深度（嚴格小於）、公里環域與計數。日期 UTC，end 為排他上界。若問題不符合，不可臆造；回覆拒絕而非分析。"},
                  {"role":"user","content":question}],
        response_format={"type":"json_schema","json_schema":{"name":"gis_plan","strict":True,"schema":schema}})
    msg = response.choices[0].message
    if getattr(msg,"refusal",None) or not msg.content:
        raise ValueError("模型拒絕或沒有提供計畫")
    plan = json.loads(msg.content)
print(json.dumps(plan, ensure_ascii=False, indent=2))
(out / "plan.json").write_text(json.dumps({"mode":mode,"question":question,"plan":plan},ensure_ascii=False,indent=2),encoding="utf-8")

## 2 安全驗證與人工核准
檢查欄位、數值、日期，拒絕任意指令。閱讀計畫確認與原問題相符，再勾選核准並執行此 cell。

In [ ]:
# @title 核准分析計畫
approved = False # @param {type:"boolean"}
assert set(plan) == set(schema["required"])
start,end = date.fromisoformat(plan["start"]),date.fromisoformat(plan["end"])
assert 0 < (end-start).days <= 366
for key,lo,hi in [("min_mag",0,10),("max_depth_km",0,700),("buffer_km",1,200)]:
    x = plan[key]
    assert type(x) in (int,float) and math.isfinite(x) and lo <= x <= hi, key
assert approved, "先核對上方計畫，再勾選 approved"
approved_plan = plan.copy()

## 3 執行資料查詢
USGS 先以固定區域、日期與規模查詢；在本機再獨立套用精確日期、規模與深度條件。保留原始資料。

In [ ]:
assert approved and plan == approved_plan
p = approved_plan
bounds = (119,21.5,123.5,26)
west,south,east,north = bounds
url = "https://earthquake.usgs.gov/fdsnws/event/1/query"
params = dict(format="geojson",starttime=p["start"],endtime=p["end"],
              minlatitude=south,maxlatitude=north,minlongitude=west,maxlongitude=east,
              minmagnitude=p["min_mag"],limit=20000,orderby="time-asc")
resp = requests.get(url,params=params,timeout=90)
resp.raise_for_status()
raw = resp.json()
assert len(raw["features"]) < 20000, "查詢達上限，縮短日期"
(out / "raw.json").write_text(json.dumps(raw))
(out / "source.json").write_text(json.dumps({"url":resp.url,"retrieved_utc":pd.Timestamp.now(tz="UTC").isoformat()},indent=2))
features = []
for f in raw["features"]:
    lon,lat,depth = f["geometry"]["coordinates"]
    a = f["properties"]
    if depth is None or a["mag"] is None: continue
    t = pd.to_datetime(a["time"],unit="ms",utc=True)
    if not pd.Timestamp(p["start"],tz="UTC") <= t < pd.Timestamp(p["end"],tz="UTC"): continue
    features.append(dict(type="Feature",geometry=dict(type="Point",coordinates=[lon,lat]),
        properties=dict(event_id=f["id"],mag=a["mag"],depth_km=depth,time=t.isoformat(),mag_type=a.get("magType"))))
assert features, "沒有候選事件，請調整條件"
fc = dict(type="FeatureCollection",features=features)
print("候選事件數",len(features))

## 4 GIS 工具執行：Select → Buffer → Render
只用允許的 RustGIS API；選取結果為空時停止，不將空選取誤當成全資料。

In [ ]:
coast_url = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_10m_coastline.geojson"
coast = gpd.clip(gpd.read_file(coast_url).to_crs(4326),box(*bounds))
coast_path = out / "coast.geojson"
coast.to_file(coast_path,driver="GeoJSON")
s = rustgis.Session()
s.set_basemap("blank",color="#DCEEF8")
s.add_geojson(str(coast_path),name="Coastline")
q = s.add_geojson(fc,name="Candidates")
where = f"mag >= {float(p['min_mag'])} AND depth_km < {float(p['max_depth_km'])}"
print("SQL:",where)
n = q.select(where)
print("RustGIS selection:",n)
assert n > 0, "無符合事件；請修改問題"
selected = q.to_geojson(selected=True)
sel = s.add_geojson(selected,name="Selected")
q.visible = False
sel.style("proportional",field="mag")
print(s.describe("buffer"))
b = s.run_tool("buffer",input=sel,distance=p["buffer_km"],units="Kilometers")
b.set_transparency(70)
s.zoom_to(bounds)
s.to_png(str(out / "agent_map.png"),width=1800,height=1800)
display(Image(filename=str(out / "agent_map.png")))
s.save(str(out / "agent.rgis"))
(out / "selected.geojson").write_text(json.dumps(selected))

## 5 獨立驗證
以 Python 原始資料重新計算事件 ID 集合，不能只比較數量。再以 WGS84 測地距離檢查環域邊界近似半徑（僅限每個非融合環域；若幾何拓撲不同則標記未驗證）。

In [ ]:
from pyproj import Geod
from shapely.geometry import shape
expected = {f["properties"]["event_id"] for f in features if f["properties"]["mag"] >= p["min_mag"] and f["properties"]["depth_km"] < p["max_depth_km"]}
actual = {f["properties"]["event_id"] for f in selected["features"]}
report = {"expected_count":len(expected),"actual_count":len(actual),
          "selection_pass":expected==actual and len(actual)==len(selected["features"]),
          "missing":sorted(expected-actual),"unexpected":sorted(actual-expected)}
geod = Geod(ellps="WGS84")
polys = b.to_geojson()["features"]
centers = [f["geometry"]["coordinates"] for f in selected["features"]]
errors = []
if len(polys)==len(centers):
    for f in polys:
        geom = shape(f["geometry"])
        if geom.geom_type != "Polygon": break
        # nearest center to polygon centroid, then sample exterior vertices
        c = geom.centroid
        center = min(centers,key=lambda xy:abs(geod.inv(c.x,c.y,*xy)[2]))
        errors.extend(abs(abs(geod.inv(center[0],center[1],x,y)[2])/1000-p["buffer_km"]) for x,y,*_ in geom.exterior.coords)
    else:
        report["buffer_max_vertex_error_km"] = max(errors) if errors else None
        report["buffer_radius_pass"] = bool(errors) and max(errors) <= max(1,0.05*p["buffer_km"])
report.setdefault("buffer_radius_pass",None)
report["buffer_note"] = "近似頂點半徑檢查，非完整拓撲或面積驗證；None 代表未驗證。"
(out / "verification.json").write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))
assert report["selection_pass"], "事件選取驗證失敗"
if report["buffer_radius_pass"] is not True:
    print("環域未通過或未完成驗證，不應據此做科學結論")

## 6 CER 與 AI 回饋
學生先寫自己的主張、證據與推理：淺層較大地震在哪裡集中？地圖與事件數能支持什麼、不能支持什麼？環域不是震度、危害或預警範圍。若選擇使用 AI 回饋，僅提供驗證報告，不要求模型捏造結果。

In [ ]:
# @title 可選 AI 解釋（live 模式）
explain = False # @param {type:"boolean"}
if explain and mode=="live":
    reply = client.chat.completions.create(model=model,messages=[
        {"role":"system","content":"以繁體中文解釋已驗證的統計，明列未驗證項目；沒有空間分區統計，不得聲稱東部或西部較集中。環域不是危害範圍。提出一個 CER 探究問題。"},
        {"role":"user","content":json.dumps({"question":question,"plan":p,"verification":report},ensure_ascii=False)}])
    print(reply.choices[0].message.content)
    (out / "ai_explanation.txt").write_text(reply.choices[0].message.content or "",encoding="utf-8")

In [ ]:
import shutil
from google.colab import files
files.download(shutil.make_archive("GIS_agent_results","zip",root_dir=out))